# Constant and Quasi-constant Features

Constant features are those that show the same value for all the observations in the dataset. Quasi-constant features are those that show the same value for the great majority of the observations.

These features provide little to no information that allows a machine learning model to discriminate or predict a target. Identifying and removing them is a crucial first step in feature selection.

In this notebook, we will demonstrate how to identify and remove these features using a realistic dataset.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold

## Create the Dataset

Let's create a dataset representing financial transactions. It will contain:
- `transaction_id`: A unique identifier (variable).
- `currency`: All transactions are in 'USD' (constant).
- `fee_structure`: FIXED for all users (constant).
- `payment_method`: Mostly 'Credit Card', but rarely 'Cash' (quasi-constant).
- `amount`: The transaction amount (variable).
- `city`: The city where the transaction occurred (variable).

In [ ]:
# Create a realistic dataset
np.random.seed(42)
n_samples = 1000

data = {
    'transaction_id': range(1, n_samples + 1),
    'currency': ['USD'] * n_samples,  # Constant
    'fee_structure': ['FIXED'] * n_samples, # Constant
    'payment_method': ['Credit Card'] * 990 + ['Cash'] * 10,  # Quasi-constant (99% Credit Card)
    'amount': np.random.rand(n_samples) * 100,
    'city': np.random.choice(['Doha', 'New York', 'London', 'Paris', 'Tokyo'], n_samples)
}

df = pd.DataFrame(data)

print("Shape of the dataframe:", df.shape)
display(df.head(10))

Shape of the dataframe: (1000, 6)


,transaction_id,currency,fee_structure,payment_method,amount,city
0,1,USD,FIXED,Credit Card,37.454012,Paris
1,2,USD,FIXED,Credit Card,95.071431,London
2,3,USD,FIXED,Credit Card,73.199394,Tokyo
3,4,USD,FIXED,Credit Card,59.865848,Doha
4,5,USD,FIXED,Credit Card,15.601864,Tokyo
5,6,USD,FIXED,Credit Card,15.599452,Doha
6,7,USD,FIXED,Credit Card,5.808361,New York
7,8,USD,FIXED,Credit Card,86.617615,Doha
8,9,USD,FIXED,Credit Card,60.111501,Paris
9,10,USD,FIXED,Credit Card,70.807258,London


## 1. Removing Constant Features

Constant features have zero variance. We can detect them using `std() == 0` for numerical features, or `nunique() == 1` for all features (including categorical).

In [ ]:
# Find constant features using nunique
constant_features = [feat for feat in df.columns if df[feat].nunique() == 1]

print("Constant features detected:", constant_features)

Constant features detected: ['currency', 'fee_structure']


In [ ]:
# Remove constant features
df_clean = df.drop(columns=constant_features)

print("Shape after removing constant features:", df_clean.shape)
display(df_clean.head())

### Using VarianceThreshold (for numerical features)

Scikit-learn's `VarianceThreshold` is a transformer that removes features with low variance. By default, it removes features with zero variance (constant features).
**Note:** `VarianceThreshold` only works with numerical data. You would need to encode categorical variables first.

## 2. Removing Quasi-constant Features

Quasi-constant features are almost constant. In our dataset, `payment_method` is 'Credit Card' for 99% of the rows.

We can identify them by checking the ratio of the most frequent value.

In [ ]:
# Check value counts for payment_method
print(df_clean['payment_method'].value_counts(normalize=True))

In [ ]:
# Define a threshold (e.g., if a value appears in more than 99% of observations)
threshold = 0.99

quasi_constant_features = []

for feat in df_clean.columns:
    # Get the ratio of the most frequent value
    top_value_ratio = df_clean[feat].value_counts(normalize=True).values[0]
    
    if top_value_ratio > threshold:
        quasi_constant_features.append(feat)

print("Quasi-constant features detected:", quasi_constant_features)

In [ ]:
# Remove quasi-constant features
df_final = df_clean.drop(columns=quasi_constant_features)

print("Shape after removing quasi-constant features:", df_final.shape)
display(df_final.head())

We have successfully reduced our feature set by identifying and removing irrelevant constant and quasi-constant features, leaving us with only the informative variables.